In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

import sys, os
os.chdir(r"C:\Users\Tysyachnyj V\Desktop\praktika_1")
from data_fetcher import load_all_data

In [2]:
def mad_score(series: pd.Series, window: int = 756) -> pd.Series:
    """
    Robust Z-score через MAD (Median Absolute Deviation).
    window=756 ≈ 3 года рабочих дней.
    score = (x - median) / (1.4826 * MAD)
    Клипируем в [-5, 5], затем масштабируем в [0, 1].
    """
    roll_median = series.rolling(window, min_periods=1).median()
    roll_mad = series.rolling(window, min_periods=1).apply(
        lambda x: np.median(np.abs(x - np.median(x))), raw=True)
    roll_mad = roll_mad.replace(0, np.nan).ffill().bfill()
    score = (series - roll_median) / (1.4826 * roll_mad)
    score = score.clip(-5, 5)
    # Нормировка в [0, 1]: 0 = нет стресса, 1 = максимум
    score_norm = (score - score.min()) / (score.max() - score.min() + 1e-9)
    return score_norm.fillna(0)


# ─────────────────────────────────────────────
#  М1 — СИГНАЛЫ
# ─────────────────────────────────────────────

def compute_m1_signals(df_m1: pd.DataFrame) -> pd.DataFrame:
    """
    Входные колонки: date, spread, ruonia
    Выходные: date, mad_score_spread, mad_score_ruonia, flag_end_of_period
    """
    df = df_m1.copy().sort_values("date").reset_index(drop=True)

    print(df)

    df["mad_score_spread"] = mad_score(df["spread"])
    df["mad_score_ruonia"] = mad_score(df["ruonia"])

    # Флаг «конец периода усреднения» — последние 3–5 дней месяца
    df["day"] = df["date"].dt.day
    df["days_in_month"] = df["date"].dt.days_in_month
    df["flag_end_of_period"] = ((df["days_in_month"] - df["day"]) <= 4).astype(int)

    # Составной сигнал М1
    df["m1_signal"] = (
        0.5 * df["mad_score_spread"] +
        0.35 * df["mad_score_ruonia"] +
        0.15 * df["flag_end_of_period"]
    )

    return df[["date", "mad_score_spread", "mad_score_ruonia",
               "flag_end_of_period", "m1_signal"]]


# ─────────────────────────────────────────────
#  М2 — СИГНАЛЫ
# ─────────────────────────────────────────────

def compute_m2_signals(df_m2: pd.DataFrame) -> pd.DataFrame:
    """
    Входные: date, cover_ratio, rate_spread
    Выходные: date, mad_score_cover, mad_score_rate_spread, flag_demand, m2_signal
    """
    df = df_m2.copy().sort_values("date").reset_index(drop=True)

    print(df)

    df["mad_score_cover"] = mad_score(df["cover_ratio"])
    df["mad_score_rate_spread"] = mad_score(df["spread_to_key"])

    # Флаг переспроса (cover > 2.0)
    df["flag_demand"] = (df["cover_ratio"] > 2.0).astype(int)

    df["m2_signal"] = (
        0.5 * df["mad_score_cover"] +
        0.35 * df["mad_score_rate_spread"] +
        0.15 * df["flag_demand"]
    )

    return df[["date", "mad_score_cover", "mad_score_rate_spread",
               "flag_demand", "m2_signal"]]


# ─────────────────────────────────────────────
#  М3 — СИГНАЛЫ
# ─────────────────────────────────────────────

def compute_m3_signals(df_m3: pd.DataFrame) -> pd.DataFrame:
    """
    Входные: date, cover_ratio, yield_spread
    Выходные: date, mad_score_cover, mad_score_yield_spread,
              flag_nedospros, flag_perespros, m3_signal
    """
    df = df_m3.copy().sort_values("date").reset_index(drop=True)
    print(df)

    # Для ОФЗ стресс = НИЗКИЙ cover (недоспрос), поэтому инвертируем
    df["cover_inv"] = -df["cover_ratio"]   # выше стресс при низком cover
    df["mad_score_cover"] = mad_score(df["cover_inv"])
    df["mad_score_yield_spread"] = mad_score(df["yield_spread"])

    df["flag_nedospros"] = (df["cover_ratio"] < 1.2).astype(int)
    df["flag_perespros"] = (df["cover_ratio"] > 2.0).astype(int)

    df["m3_signal"] = (
        0.55 * df["mad_score_cover"] +
        0.30 * df["mad_score_yield_spread"] +
        0.15 * df["flag_nedospros"]
    )

    return df[["date", "mad_score_cover", "mad_score_yield_spread",
               "flag_nedospros", "flag_perespros", "m3_signal"]]


# ─────────────────────────────────────────────
#  М4 — СИГНАЛЫ (мультипликатор)
# ─────────────────────────────────────────────

def compute_m4_signals(df_m4: pd.DataFrame) -> pd.DataFrame:
    """М4 — детерминированный, выдаёт seasonal_factor и флаги."""
    return df_m4.copy()


# ─────────────────────────────────────────────
#  М5 — СИГНАЛЫ
# ─────────────────────────────────────────────

def compute_m5_signals(df_m5: pd.DataFrame) -> pd.DataFrame:
    """
    Входные: date, structural_balance, treasury_balance, delta_weekly
    Выходные: date, mad_score_structural, mad_score_delta, flag_budget_drain, m5_signal
    """
    df = df_m5.copy().sort_values("date").reset_index(drop=True)
    print(df)

    # Стресс = падение структурного баланса (более отрицательный = хуже)
    df["structural_inv"] = -df["structural_balance"]
    df["delta_inv"] = -df["delta_weekly"].fillna(0)

    df["mad_score_structural"] = mad_score(df["structural_inv"])
    df["mad_score_delta"] = mad_score(df["delta_inv"])

    # Флаг резкого оттока: delta < -300 млрд за неделю (в трлн: < -0.3)
    threshold = df["delta_weekly"].quantile(0.10)  # нижние 10%
    df["flag_budget_drain"] = (df["delta_weekly"] < threshold).astype(int)

    df["m5_signal"] = (
        0.5 * df["mad_score_structural"] +
        0.35 * df["mad_score_delta"] +
        0.15 * df["flag_budget_drain"]
    )

    return df[["date", "mad_score_structural", "mad_score_delta",
               "flag_budget_drain", "m5_signal"]]


# ─────────────────────────────────────────────
#  АГРЕГАЦИЯ В ЕДИНЫЙ ДАТАФРЕЙМ
# ─────────────────────────────────────────────

def build_feature_matrix(signals: dict) -> pd.DataFrame:
    """
    Объединяем сигналы всех модулей в единый датафрейм по бизнес-дням.
    Нечастые данные (М2, М3) — forward-fill.
    """
    # М1 — база (ежемесячно, но у нас ежедневные данные)
    base = signals["m1"][["date", "m1_signal", "mad_score_spread",
                           "mad_score_ruonia", "flag_end_of_period"]].copy()

    # М2 — ffill (еженедельно)
    m2 = signals["m2"][["date", "m2_signal", "mad_score_cover",
                          "mad_score_rate_spread", "flag_demand"]].copy()

    # М3 — ffill (2x в неделю)
    m3 = signals["m3"][["date", "m3_signal", "flag_nedospros",
                          "flag_perespros"]].copy()

    # М4 — ежедневно
    m4 = signals["m4"][["date", "tax_week_flag", "end_of_month_flag",
                          "end_of_quarter_flag", "seasonal_factor"]].copy()

    # М5 — ежедневно
    m5 = signals["m5"][["date", "m5_signal", "mad_score_structural",
                          "flag_budget_drain"]].copy()

    df = base.copy()
    for other in [m2, m3, m4, m5]:
        df = pd.merge_asof(
            df.sort_values("date"),
            other.sort_values("date"),
            on="date", direction="backward"
        )

    df = df.ffill().fillna(0)
    df = df.sort_values("date").reset_index(drop=True)
    return df


# ─────────────────────────────────────────────
#  LSI АГРЕГАЦИЯ
# ─────────────────────────────────────────────

# Базовые веса модулей (обоснование: М2 наиболее оперативен,
# М1 и М3 — важные структурные индикаторы, М4 — мультипликатор,
# М5 — опережающий сигнал)
MODULE_WEIGHTS = {
    "m1_signal": 0.20,
    "m2_signal": 0.25,
    "m3_signal": 0.20,
    "m5_signal": 0.15,
    # М4 входит через seasonal_factor как мультипликатор
}

FEATURE_COLS = ["m1_signal", "m2_signal", "m3_signal", "m5_signal",
                "mad_score_spread", "mad_score_ruonia",
                "mad_score_cover", "mad_score_rate_spread",
                "flag_demand", "flag_nedospros", "flag_budget_drain",
                "tax_week_flag", "end_of_quarter_flag"]


def compute_lsi_weighted(df: pd.DataFrame) -> pd.DataFrame:
    """
    Метод 1 (базовый): взвешенная сумма с seasonal_factor-мультипликатором.
    Простой, интерпретируемый, не требует разметки.
    """
    df = df.copy()

    raw = (
        MODULE_WEIGHTS["m1_signal"] * df["m1_signal"] +
        MODULE_WEIGHTS["m2_signal"] * df["m2_signal"] +
        MODULE_WEIGHTS["m3_signal"] * df["m3_signal"] +
        MODULE_WEIGHTS["m5_signal"] * df["m5_signal"]
    )

    # Применяем seasonal_factor как мультипликатор (М4)
    seasonal = df.get("seasonal_factor", pd.Series(1.0, index=df.index))
    raw_seasonal = raw * seasonal

    # Нормировка в [0, 100]
    lsi_raw = (raw_seasonal - raw_seasonal.min()) / \
              (raw_seasonal.max() - raw_seasonal.min() + 1e-9) * 100

    df["lsi"] = lsi_raw.clip(0, 100)

    # Вклад каждого модуля (в единицах LSI)
    total = sum(MODULE_WEIGHTS.values())
    df["contrib_m1"] = MODULE_WEIGHTS["m1_signal"] / total * df["m1_signal"] * 100
    df["contrib_m2"] = MODULE_WEIGHTS["m2_signal"] / total * df["m2_signal"] * 100
    df["contrib_m3"] = MODULE_WEIGHTS["m3_signal"] / total * df["m3_signal"] * 100
    df["contrib_m5"] = MODULE_WEIGHTS["m5_signal"] / total * df["m5_signal"] * 100
    df["contrib_m4"] = (seasonal - 1.0) * 20  # вклад сезонного множителя

    return df


def compute_lsi_ml(df: pd.DataFrame) -> pd.DataFrame:
    """
    Метод 2 (ML): Ridge-регрессия с псевдо-разметкой на стресс-эпизодах.
    Выбор Ridge обоснован: линейная модель → прозрачные коэффициенты
    (аналог явной формулы с весами), устойчива к мультиколлинеарности сигналов.
    Интерпретация: feature_importances = coef_ Ridge.
    """
    df = df.copy()
    feature_cols = [c for c in FEATURE_COLS if c in df.columns]

    # Псевдо-разметка: стресс-эпизоды получают высокий таргет
    stress_periods = [
        ("2014-12-10", "2015-01-20", 95),
        ("2022-02-24", "2022-04-30", 90),
        ("2023-08-01", "2023-10-01", 80),
    ]
    df["target"] = 20.0  # базовый уровень

    for start, end, val in stress_periods:
        mask = (df["date"] >= start) & (df["date"] <= end)
        df.loc[mask, "target"] = val

    # Обучение
    X = df[feature_cols].fillna(0).values
    y = df["target"].values

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    model = Ridge(alpha=1.0)
    model.fit(X_scaled, y)

    lsi_raw = model.predict(X_scaled)
    lsi_norm = (lsi_raw - lsi_raw.min()) / (lsi_raw.max() - lsi_raw.min() + 1e-9) * 100
    df["lsi"] = lsi_norm.clip(0, 100)

    # Вклад модулей через суммирование коэффициентов по группам
    coefs = dict(zip(feature_cols, model.coef_))

    def _group_contrib(cols, x_cols, X_s, coefs):
        idx = [x_cols.index(c) for c in cols if c in x_cols]
        if not idx:
            return 0
        return float(np.mean(np.abs([coefs[x_cols[i]] * X_s[:, i].mean() for i in idx])))

    m1_cols = ["m1_signal", "mad_score_spread", "mad_score_ruonia"]
    m2_cols = ["m2_signal", "mad_score_cover", "mad_score_rate_spread", "flag_demand"]
    m3_cols = ["m3_signal", "flag_nedospros"]
    m4_cols = ["tax_week_flag", "end_of_quarter_flag"]
    m5_cols = ["m5_signal", "flag_budget_drain"]

    groups = [m1_cols, m2_cols, m3_cols, m4_cols, m5_cols]
    raw_c = [_group_contrib(g, feature_cols, X_scaled, coefs) for g in groups]
    total_c = sum(raw_c) + 1e-9

    df["contrib_m1"] = raw_c[0] / total_c * df["lsi"]
    df["contrib_m2"] = raw_c[1] / total_c * df["lsi"]
    df["contrib_m3"] = raw_c[2] / total_c * df["lsi"]
    df["contrib_m4"] = raw_c[3] / total_c * df["lsi"]
    df["contrib_m5"] = raw_c[4] / total_c * df["lsi"]

    # Сохраняем коэффициенты для отчёта
    df.attrs["ridge_coefs"] = coefs
    df.attrs["feature_cols"] = feature_cols
    df.attrs["model"] = model
    df.attrs["scaler"] = scaler

    return df


def add_status(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляем статус: ЗЕЛЁНЫЙ / ЖЁЛТЫЙ / КРАСНЫЙ."""
    conditions = [
        df["lsi"] < 40,
        (df["lsi"] >= 40) & (df["lsi"] < 70),
        df["lsi"] >= 70
    ]
    choices = ["🟢 НОРМА", "🟡 ВНИМАНИЕ", "🔴 СТРЕСС"]
    df["status"] = np.select(conditions, choices, default="🟢 НОРМА")
    df["status_color"] = np.select(conditions, ["green", "orange", "red"],
                                    default="green")
    return df

In [3]:
def run_signal_engine(raw_data: dict, method: str = "weighted") -> dict:
    """
    Запускает полный пайплайн расчёта сигналов.
    method: 'weighted' | 'ml'
    """
    print(f"\n[Signal Engine] Метод агрегации: {method}")

    signals = {
        "m1": compute_m1_signals(raw_data["m1"]),
        "m2": compute_m2_signals(raw_data["m2"]),
        "m3": compute_m3_signals(raw_data["m3"]),
        "m4": compute_m4_signals(raw_data["m4"]),
        "m5": compute_m5_signals(raw_data["m5"]),
    }

    df_features = build_feature_matrix(signals)
    print(f"[Signal Engine] Feature matrix: {df_features.shape}")

    if method == "ml":
        df_lsi = compute_lsi_ml(df_features)
    else:
        df_lsi = compute_lsi_weighted(df_features)

    df_lsi = add_status(df_lsi)


    return {
        "signals": signals,
        "features": df_features,
        "lsi": df_lsi,
    }

In [4]:
if __name__ == "__main__":
    data = load_all_data()
    results = run_signal_engine(data, method="weighted")
    print(f"\nПоследнее значение LSI: {results['lsi']['lsi'].iloc[-1]:.1f}")
    print(f"Статус: {results['lsi']['status'].iloc[-1]}")

Загрузка данных для всех модулей...
RUONIA: данные загружены с сайта ЦБ
М1: данные загружены с сайта ЦБ
Этап 1: получение списка дат с аукционами...
Найдено 101 уникальных дат с аукционами
Этап 2: загрузка детальных данных по каждой дате...
2016-01-12
2016-01-19
2016-01-26
2016-02-02
2016-02-09
2016-02-16
2016-02-20
2016-03-01
2016-03-04
2016-03-15
2016-03-22
2016-03-29
2016-04-05
2016-04-12
2016-04-19
2016-04-26
2016-04-29
2016-05-10
2016-05-17
2016-05-24
2016-05-31
2016-06-07
2016-06-14
2016-06-21
2016-06-28
2016-07-05
2016-07-12
2016-07-19
2016-07-26
2016-08-02
2016-08-16
2016-08-23
2016-09-20
2020-03-30
2022-03-01
2022-03-09
2022-03-15
2022-03-22
2022-03-29
2022-04-05
2022-04-12
2022-04-19
2022-04-26
2022-05-04
2022-05-11
2025-04-15
2025-04-22
2025-04-29
2025-05-06
2025-05-13
2025-05-20
2025-05-27
2025-06-03
2025-06-10
2025-06-17
2025-06-24
2025-07-01
2025-07-08
2025-07-15
2025-07-22
2025-07-29
2025-08-05
2025-08-12
2025-08-19
2025-08-26
2025-09-02
2025-09-09
2025-09-16
2025-09-23


In [5]:
import requests
from io import StringIO

REPO_URL = "https://www.cbr.ru/hd_base/repo/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://www.cbr.ru/"
}

def _build_cbr_params(start, end):
    return {
        "UniDbQuery.Posted": "True",
        "UniDbQuery.From": pd.Timestamp(start).strftime("%d.%m.%Y"),
        "UniDbQuery.To": pd.Timestamp(end).strftime("%d.%m.%Y"),
    }

def _safe_read_html(html_text):
    """Безопасно парсит HTML, возвращает список таблиц или пустой список"""
    try:
        return pd.read_html(StringIO(html_text), decimal=",", thousands=" ")
    except Exception:
        return []

def _to_num(val):
    """Преобразует значение в число, обрабатывая пробелы и запятые"""
    try:
        return float(str(val).replace(" ", "").replace(",", "."))
    except:
        return float("nan")
    
def _find_col(df, *keywords):
    """Ищет колонку, у которой хотя бы один уровень содержит все keywords."""
    kws = [k.lower() for k in keywords]
    for i, col in enumerate(df.columns):
        levels = [str(c).lower() for c in (col if isinstance(col, tuple) else [col])]
        full = " ".join(levels)
        if all(k in full for k in kws):
            return i
    return None

def _extract_records_from_table(df, fallback_date):
    """Извлекает записи из таблицы одного дня"""
    records = []
    idx_date   = _find_col(df, "дата")
    idx_demand = _find_col(df, "спроса", "заявок") or _find_col(df, "объем спроса")
    idx_place  = _find_col(df, "заключенных", "сделок") or _find_col(df, "размещения")
    idx_cutoff = _find_col(df, "ставка", "отсечения")
    idx_avg    = _find_col(df, "средневзвешенная", "ставка")
    idx_type   = _find_col(df, "тип", "операции")

    if idx_date is None or idx_demand is None or idx_place is None:
        return records

    for row_idx in range(df.shape[0]):
        row = df.iloc[row_idx]
        try:
            date_val = pd.to_datetime(row.iloc[idx_date], dayfirst=True)
        except:
            date_val = fallback_date

        demand = _to_num(row.iloc[idx_demand])
        placement = _to_num(row.iloc[idx_place])
        if pd.isna(demand) or pd.isna(placement):
            continue

        rec = {
            "date":          date_val,
            "demand":        demand,
            "placement":     placement,
            "cut_off_rate":  _to_num(row.iloc[idx_cutoff]) if idx_cutoff is not None else None,
            "avg_rate":      _to_num(row.iloc[idx_avg]) if idx_avg is not None else None,
            "auction_type":  str(row.iloc[idx_type]).strip() if idx_type is not None else None
        }
        records.append(rec)
    return records

def _parse_vertical_day_table(df, date):
    """
    Парсит вертикальную таблицу одного дня.
    Ожидается структура: колонка 0 - название параметра, колонка 1 - значение.
    Разделителем аукционов служит строка с 'Тип аукциона' или пустая строка.
    """
    records = []
    current = {}
    # Определяем индексы колонок (могут быть MultiIndex)
    col_label = 0
    col_value = 1
    if df.shape[1] < 2:
        return records

    for _, row in df.iterrows():
        label = str(row.iloc[col_label]).strip().lower()
        value = row.iloc[col_value]

        # Пропускаем пустые строки и заголовки
        if label in ('', 'nan', 'параметр', 'значение'):
            if current and current.get('demand') is not None:
                records.append(current)
                current = {}
            continue

        # Начало нового аукциона
        if 'тип аукциона' in label:
            if current and current.get('demand') is not None:
                records.append(current)
                current = {}
            current = {'auction_type': str(value).strip()}
            continue

        # Извлечение параметров по ключевым словам
        if 'объем спроса' in label and 'заключенных' not in label:
            current['demand'] = _to_num(value)
        elif ('общий объем заключенных сделок' in label or 
              ('объем заключенных сделок' in label and 'в рамках лимита' not in label)):
            current['placement'] = _to_num(value)
        elif 'ставка отсечения' in label:
            current['cut_off_rate'] = _to_num(value)
        elif 'средневзвешенная ставка' in label:
            current['avg_rate'] = _to_num(value)
        elif 'срок' in label and 'дни' in label:
            term_val = _to_num(value)
            current['term'] = int(term_val) if pd.notna(term_val) else None
        # можно добавить другие параметры при необходимости

    # Последний собранный аукцион
    if current and current.get('demand') is not None:
        records.append(current)

    # Добавляем дату
    for rec in records:
        rec['date'] = date
    return records

date = pd.Timestamp("2025-08-19")
all_records = []
params_day = _build_cbr_params(date, date)
params_day['UniDbQuery.ShowAll'] = '1'
params_day['UniDbQuery.P1'] = '5'
r = requests.get(REPO_URL, params=params_day, headers=HEADERS, timeout=20)
r.raise_for_status()
tables = _safe_read_html(r.text)
if tables:
    df_day = tables[0]
    if df_day.shape[0] > 1:
        # парсим вертикальную таблицу дня
        records = _parse_vertical_day_table(df_day, pd.Timestamp(date))
        all_records.extend(records)

all_records

[{'demand': 644023.2,
  'placement': 310000.0,
  'cut_off_rate': 18.1117,
  'avg_rate': 18.2385,
  'term': 7,
  'date': Timestamp('2025-08-19 00:00:00')}]

In [6]:
df = pd.DataFrame({"B": [i for i in range(10)]})
df

,B
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,7
8,8
9,9


In [ ]:
df.rolling(5, min_periods=2).median()

,B
0,NaN
1,0.5
2,1.0
3,1.5
4,2.0
5,3.0
6,4.0
7,5.0
8,6.0
9,7.0
